<a href="https://colab.research.google.com/github/dptel22/Nosh-Robotics/blob/main/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pull SAM 3 model

In [1]:
!git clone https://github.com/facebookresearch/sam3.git
!pip install -q -e ./sam3


fatal: destination path 'sam3' already exists and is not an empty directory.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sam3 (pyproject.toml) ... done


Hugging face login

In [1]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

Load the data

In [3]:
#!unzip -q "/content/egg.v3i.folder.zip" -d /content/

replace /content/egg.v3i.folder/README.dataset.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/egg.v3i.folder/README.roboflow.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/egg.v3i.folder/test/egg/not_damaged_101_jpg.rf.38605ebedec1a67bbdfd051be8fcbf76.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/egg.v3i.folder/test/egg/not_damaged_125_jpg.rf.07d5daf0e7b3d3fb818aef3a0e914c05.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/egg.v3i.folder/test/egg/not_damaged_14_jpg.rf.2c71134214876a79eaa55e462b6299b7.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/egg.v3i.folder/test/egg/not_damaged_152_jpg.rf.7be9845f57433bfee52d1fdd1b448833.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/egg.v3i.folder/test/egg/not_damaged_153_jpg.rf.e3220332053fdb10558c6c36c01849cf.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/egg.v3i.folder/test/egg/not_damaged_161_jpg.rf.5f79f00963395a52b24de647dd6528a9.jpg? [y]e

In [4]:
from pathlib import Path

# Define the simple path to your extracted dataset
dataset_dir = Path("/content/egg.v3i.folder")

# Find all images in the dataset folder
valid_extensions = {".jpg", ".jpeg", ".png", ".webp"}
all_images = [p for p in dataset_dir.rglob("*") if p.is_file() and p.suffix.lower() in valid_extensions]

# Print loading summary
print(f"Dataset Folder: {dataset_dir.absolute()}")
print(f"Total Images Loaded: {len(all_images)}")
if all_images:
    print("Example Image Path:", all_images[0])
else:
    print("No images found. Please verify the unzip command executed fully.")

Dataset Folder: /content/egg.v3i.folder
Total Images Loaded: 160
Example Image Path: /content/egg.v3i.folder/train/egg/not_damaged_128_jpg.rf.6858d59b27b6c88c6e18cb85dbe6fff7.jpg


Check the environment

In [2]:
import sys
import numpy as np
import torch

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("NumPy test:", np.random.rand(3))

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("SAM 3 imports successful")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 1.26.4
PyTorch: 2.11.0+cu130
CUDA available: True
NumPy test: [0.7505056  0.19245184 0.13363659]


/content/sam3/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/content/sam3/sam3/model/model_misc.py:71: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


SAM 3 imports successful


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Load SAM 3

In [3]:
import os
device = "cuda" if torch.cuda.is_available() else "cpu"

# Manually define the BPE vocabulary path to bypass the pkg_resources error
bpe_path = "/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
if not os.path.exists(bpe_path):
    # fallback to local relative path if needed
    bpe_path = "sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"

model = build_sam3_image_model(bpe_path=bpe_path, device=device)
model.eval()

processor = Sam3Processor(
    model,
    device=device,
    confidence_threshold=0.30,
)

print("SAM 3 loaded on:", device)

config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

sam3.pt: reconstructing file:   0%|          |  0.00B / 3.45GB            

sam3.pt: downloading bytes:           |  0.00B            

SAM 3 loaded on: cuda


Define segmentation and mask processing

In [5]:
from PIL import Image
import numpy as np
import torch

def segment_image(path):
    image = Image.open(path).convert("RGB")

    with torch.inference_mode():
        state = processor.set_image(image)
        result = processor.set_text_prompt(
            state=state,
            prompt="egg",
        )

    masks = result["masks"].detach().cpu().numpy()
    if masks.ndim == 4:
        masks = masks[:, 0]  # (N, 1, H, W) -> (N, H, W)

    masks = masks.astype(bool)
    scores = result["scores"].detach().cpu().numpy().reshape(-1)

    return image, masks, scores


def select_masks(masks, scores, min_area=0.002, max_area=0.98, iou_threshold=0.70):
    """Keep plausible-sized masks and remove near-identical duplicates."""
    if len(masks) == 0:
        return []

    valid = [
        i for i, mask in enumerate(masks)
        if min_area <= mask.mean() <= max_area
    ]

    kept = []

    for i in sorted(valid, key=lambda j: scores[j], reverse=True):
        duplicate = False

        for j in kept:
            intersection = np.logical_and(masks[i], masks[j]).sum()
            union = np.logical_or(masks[i], masks[j]).sum()

            if union and intersection / union > iou_threshold:
                duplicate = True
                break

        if not duplicate:
            kept.append(i)

    return kept


def make_overlay(image, masks, scores, kept):
    """Colour the egg masks over the original image."""
    result = np.asarray(image).copy()

    colours = [
        (255, 60, 60),
        (40, 220, 100),
        (40, 140, 255),
        (255, 190, 30),
        (210, 70, 255),
    ]

    for instance_id, i in enumerate(kept):
        mask = masks[i]
        colour = np.array(colours[instance_id % len(colours)], dtype=np.uint8)

        result[mask] = (
            0.55 * result[mask] + 0.45 * colour
        ).astype(np.uint8)

        # Mark the mask boundary without requiring OpenCV.
        eroded = np.zeros_like(mask)
        eroded[1:-1, 1:-1] = (
            mask[1:-1, 1:-1]
            & mask[:-2, 1:-1]
            & mask[2:, 1:-1]
            & mask[1:-1, :-2]
            & mask[1:-1, 2:]
        )
        result[mask & ~eroded] = colour

    return Image.fromarray(result)

Test on one image

In [6]:
from IPython.display import display

sample_path = image_paths[0]

image, masks, scores = segment_image(sample_path)
kept = select_masks(masks, scores)

print("Image:", sample_path.name)
print("Raw detections:", len(masks))
print("Masks after duplicate removal:", len(kept))
print("Confidence scores:", [round(float(scores[i]), 3) for i in kept])

display(image)
display(make_overlay(image, masks, scores, kept))

NameError: name 'image_paths' is not defined